# Tutorial 1: Environments and Neural Networks

**zrth** extracts symbolic reactive modules from Python classes. You write a standard [Gymnasium](https://gymnasium.farama.org/) environment or a [PyTorch](https://pytorch.org/) neural network, then wrap it with `Wrapper()` or `Module()` — zrth analyzes it and produces a formal model with typed wires and terms that can be used for verification or training.

This tutorial walks through:
1. Defining an environment (plain `gym.Env`) and wrapping it with `zrth.gym.Wrapper`
2. Defining a neural network (plain `nn.Module`) and wrapping it with `zrth.torch.Module`
3. Inspecting the extracted modules
4. Training a ranking function to prove a liveness property
5. Formally verifying the ranking conditions with Z3

## Defining an environment

Write a standard [Gymnasium](https://gymnasium.farama.org/) environment by subclassing `gym.Env`:

- **`__init__`**: set `action_space`, `observation_space`, any parameters (e.g. `self.y0`), and state variables (e.g. `self.x`)
- **`reset`**: initialize all state variables, return `(observation, info)`
- **`step`**: update state given an action, return `(observation, reward, terminated, truncated, info)`

**Important:** state variables must also be assigned in `__init__` so the analyzer can infer their types. For example, even though `reset` sets `self.x = 0.0`, you should also write `self.x = 0.0` in `__init__`.

Below is a counter system with three variables $x$, $y$, $z$. The variable $x$ increments while $x < y \lor x < z$, and resets to $0$ otherwise. We want to show that $x = y \lor x = z$ must occur infinitely often.

In [1]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth.gym import Wrapper
from zrth.torch import Module


class CounterEnv(gym.Env):
    """Counter environment: x increments while x < y or x < z, resets otherwise."""

    def __init__(self, y0=5.0, z0=3.0):
        super().__init__()

        self.action_space = spaces.Discrete(1)
        self.observation_space = spaces.Box(low=-1e6, high=1e6, shape=(1,))

        self.y0 = y0
        self.z0 = z0

        # Also set state variables here so the analyzer can infer their dtype
        self.x = 0.0
        self.y = y0
        self.z = z0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.x = 0.0
        self.y = self.y0
        self.z = self.z0

        observation = self.x
        return observation, {}

    def step(self, action):
        if self.x < self.y or self.x < self.z:
            self.x = self.x + 1.0
        else:
            self.x = 0.0

        # y and z remain constant
        self.y = self.y
        self.z = self.z

        at_target = self.x == self.y or self.x == self.z
        reward = 1.0 if at_target else 0.0
        terminated = at_target
        truncated = False
        observation = self.x
        return observation, reward, terminated, truncated, {}

## Wrapping and inspection

When you call `Wrapper(counter_instance)`, zrth analyzes the class's `__init__`, `reset`, and `step` methods. It:
1. Infers action/observation types from the gymnasium spaces
2. Classifies `self.*` attributes as **private** (read+write state) or **parameters** (read-only constants)
3. Creates typed wire pairs for each variable and signal
4. Converts `reset` into **init** terms and `step` into **update** terms

Print the extracted module to see the result.

In [2]:
counter = CounterEnv(y0=5.0, z0=3.0)
wrapped_counter = Wrapper(counter)
print(wrapped_counter)

module
  external
    w8, w9 : Float(1)
  interface
    w10, w11 : Float(1)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
  private
    w0, w1 : Float(1)
    w2, w3 : Float(1)
    w4, w5 : Float(1)
  atom controls w1, w3, w5, w11, w13, w15, w17 reads w0, w2, w4
  init
    Tensor([5]) w6; 
    Tensor([3]) w7; 
    Tensor([0]) w18; 
    Id w3; w18
    Id w5; w6
    Id w1; w7
    Id w11; w18
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([5]) w6; 
    Tensor([3]) w7; 
    Lt w19; w2, w4
    Lt w20; w2, w0
    Tensor([0]) w21; 
    Tensor([1]) w22; 
    Ite w23; w19, w22, w20
    Tensor([1]) w24; 
    Add w25; w2, w24
    Tensor([0]) w26; 
    Ite w27; w23, w25, w26
    Id w3; w27
    Id w5; w4
    Id w1; w0
    Eq w28; w27, w4
    Eq w29; w27, w0
    Tensor([0]) w30; 
    Tensor([1]) w31; 
    Ite w32; w28, w31, w29
    Tensor([1]) w33; 
    Tensor([0]) w34; 
    Ite w35; w32, w33, w34
    Tensor([0]) w36; 
    Id w11; w27
 

## Reading the output

The printed module shows:

- **external**: input wires — here, the action (unused by this environment, but still present)
- **interface**: output wires — observation, reward, terminated, truncated
- **private**: internal state — wire pairs `wN, wM` for x, y, z (each pair is `latched, next`)
- **init**: terms from `reset` — how each wire is initialized
- **update**: terms from `step` — how each wire is updated, including `Ite` (if-then-else), `Lt`, `Eq`, `Add`

## Defining a neural network

Write a standard `nn.Module`, then wrap it with `Module(instance)` to extract a symbolic module:

- **`__init__`**: define `nn.Linear` layers
- **`forward`**: standard PyTorch forward pass

zrth extracts the architecture as a **combinatorial** (stateless) module. The resulting module has:
- **extl** (external input) wire pair — sized from the first layer's `in_features`
- **intf** (interface output) wire pair — sized from the last layer's `out_features`

The ranking function below maps $(x, y, z) \in \mathbb{R}^3$ to a scalar. It is used in formal verification to prove that the liveness property $x = y \lor x = z$ holds infinitely often — not a controller, so we don't compose it with the counter.

In [3]:
class RankingNN(nn.Module):
    """Ranking function: R(x, y, z) -> scalar.

    Architecture: [3] -> 2 -> [1]
    """

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 2)
        self.fc2 = nn.Linear(2, 1)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        return self.fc2(x)

Wrap and inspect. Note that the module has no `private` wires (it's stateless) and the `init`/`update` blocks are identical (combinatorial — same computation every step).

In [4]:
ranking = RankingNN()
wrapped_ranking = Module(ranking)
print(wrapped_ranking)

module
  external
    w37, w38 : Float(3)
  interface
    w39, w40 : Float(1)
  atom controls w40 awaits w38
  init
    Tensor([0.4910790026187897 -0.08310495316982269 -0.4675257205963135 0.4608395993709564 -0.03646361827850342 ...]) w41; 
    Tensor([-0.4849992096424103 0.06335223466157913]) w42; 
    Linear w43; w38, w41, w42
    ReLU w44; w43
    Tensor([-0.48408469557762146 -0.09584648162126541]) w45; 
    Tensor([0.6993432641029358]) w46; 
    Linear w47; w44, w45, w46
    Id w40; w47
  update
    Tensor([0.4910790026187897 -0.08310495316982269 -0.4675257205963135 0.4608395993709564 -0.03646361827850342 ...]) w41; 
    Tensor([-0.4849992096424103 0.06335223466157913]) w42; 
    Linear w43; w38, w41, w42
    ReLU w44; w43
    Tensor([-0.48408469557762146 -0.09584648162126541]) w45; 
    Tensor([0.6993432641029358]) w46; 
    Linear w47; w44, w45, w46
    Id w40; w47



## Training the ranking function

A **ranking function** $R: \mathcal{S} \to \mathbb{R}$ can be used to prove that a liveness property holds infinitely often. The key conditions are:

1. $R(s)$ is bounded from below (i.e. $R(s) \geq c$ for some constant $c$)
2. $R(s) = 0$ when the liveness property holds (here: $x = y \lor x = z$)
3. $R(s') < R(s)$ whenever the property does *not* hold (the ranking strictly decreases)

If such an $R$ exists, the property must hold infinitely often — otherwise $R$ would decrease forever without bound, contradicting condition 1.

We train the `RankingNN` to approximate these conditions by:
1. Simulating the counter for $N$ steps (it's a fully functional `gym.Env`)
2. At each step, computing $R(x, y, z)$ and $R(x', y', z')$
3. Penalizing violations: $R$ should decrease at non-target states and be near zero at target states

In [5]:
optimizer = torch.optim.Adam(wrapped_ranking.parameters(), lr=0.01)
margin = 0.1
n_epochs = 300
n_steps = 20

for epoch in range(n_epochs):
    # Simulate the counter to collect a trajectory of (x, y, z) states.
    # CounterEnv is a full gym.Env — reset() and step() work directly.
    wrapped_counter.reset()

    states = []
    for _ in range(n_steps):
        states.append((wrapped_counter.x, wrapped_counter.y, wrapped_counter.z))
        wrapped_counter.step(0)  # Discrete(1) action space — always 0

    # Compute ranking loss over consecutive state pairs
    loss = torch.tensor(0.0)
    for i in range(len(states) - 1):
        s = torch.tensor(states[i], dtype=torch.float32)
        s_next = torch.tensor(states[i + 1], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze()
        r_next = wrapped_ranking(s_next.unsqueeze(0)).squeeze()

        x, y, z = states[i]
        at_target = (x == y) or (x == z)

        if not at_target:
            # R must strictly decrease by at least `margin`
            loss = loss + torch.relu(r_next - r + margin)
            # Push R upward (helps with bounded-from-below condition)
            loss = loss + torch.relu(-r)
        else:
            # R should be ~0 at target states
            loss = loss + r ** 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.4f}")

epoch   0  loss = 3.3673
epoch  50  loss = 0.0675
epoch 100  loss = 0.0641
epoch 150  loss = 0.0615
epoch 200  loss = 0.0666
epoch 250  loss = 0.0618


## Evaluating the trained ranking function

After training, we evaluate $R$ along a trajectory. At non-target states, $R$ should decrease on each step. At target states ($x = y$ or $x = z$), $R$ should be near zero. These are approximate — formal verification follows below.

In [6]:
wrapped_counter.reset()

print(f"{'step':>4}  {'x':>5} {'y':>5} {'z':>5}  {'R(s)':>8}  {'target':>6}")
print("-" * 45)

with torch.no_grad():
    for step in range(20):
        x, y, z = wrapped_counter.x, wrapped_counter.y, wrapped_counter.z
        s = torch.tensor([x, y, z], dtype=torch.float32)
        r = wrapped_ranking(s.unsqueeze(0)).squeeze().item()
        at_target = (x == y) or (x == z)
        print(f"{step:4d}  {x:5.1f} {y:5.1f} {z:5.1f}  {r:8.4f}  {'  *' if at_target else ''}")
        wrapped_counter.step(0)

step      x     y     z      R(s)  target
---------------------------------------------
   0    0.0   5.0   3.0    0.4349  
   1    1.0   5.0   3.0    0.3273  
   2    2.0   5.0   3.0    0.2196  
   3    3.0   5.0   3.0    0.1120    *
   4    4.0   5.0   3.0    0.0044  
   5    5.0   5.0   3.0   -0.1032    *
   6    0.0   5.0   3.0    0.4349  
   7    1.0   5.0   3.0    0.3273  
   8    2.0   5.0   3.0    0.2196  
   9    3.0   5.0   3.0    0.1120    *
  10    4.0   5.0   3.0    0.0044  
  11    5.0   5.0   3.0   -0.1032    *
  12    0.0   5.0   3.0    0.4349  
  13    1.0   5.0   3.0    0.3273  
  14    2.0   5.0   3.0    0.2196  
  15    3.0   5.0   3.0    0.1120    *
  16    4.0   5.0   3.0    0.0044  
  17    5.0   5.0   3.0   -0.1032    *
  18    0.0   5.0   3.0    0.4349  
  19    1.0   5.0   3.0    0.3273  


## Formal verification with Z3

Training gives us a *candidate* ranking function, but empirical checks on a single trajectory cannot guarantee that the conditions hold for *every* reachable state.

We can use an SMT solver ([Z3](https://github.com/Z3Prover/z3)) to verify the ranking conditions exhaustively. The idea: **negate** a condition and ask Z3 whether a counterexample exists. If Z3 returns `unsat`, no counterexample exists and the condition holds universally.

`zrth.smt.z3` translates module terms into Z3 expressions — the same term IR that `zrth.eval` executes with PyTorch tensors. Below we verify conditions 1 (bounded from below) and 3 (strict decrease).

### Building the symbolic state

First, create Z3 real variables for the counter's state $(x, y, z)$ and execute the counter's update block symbolically. This gives us $x'$ — the next-state value of $x$ — as a Z3 expression.

In [7]:
import z3
from zrth.smt.z3 import z3_eval_itype

# Z3 variables for the counter state
x_var, y_var, z_var = z3.Reals('x y z')

# Map counter's latched private wires to Z3 variables
wire_names = object.__getattribute__(wrapped_counter, '_wire_names')
state = {
    wire_names['x'][0]: [x_var],  # x latched
    wire_names['y'][0]: [y_var],  # y latched
    wire_names['z'][0]: [z_var],  # z latched
}

# Execute the counter's update block term-by-term.
# z3_eval_itype translates a single term's IType into Z3 expressions;
# the loop over atoms and terms mirrors what zrth.eval does with PyTorch.
for atom in wrapped_counter.atoms:
    for term in atom.update:
        read = [state[w] for w in term.read]
        results = z3_eval_itype(term.itype, read)
        for w, val in zip(term.write, results):
            state[w] = val

x_next = state[wire_names['x'][1]][0]
print("x' =", x_next)

x' = If(If(x < y, True, x < z), x + 1, 0)


### Building the symbolic ranking function

Execute the ranking module's terms with Z3 to get $R(s)$ and $R(s')$ as Z3 expressions. The trained weights become exact rational constants in Z3.

In [8]:
extl = list(wrapped_ranking.extl)
intf = list(wrapped_ranking.intf)

def eval_ranking(input_vars):
    """Evaluate the ranking module's terms with Z3, returning R as a Z3 expression."""
    r_state = {extl[0][1]: input_vars}
    for atom in wrapped_ranking.atoms:
        for term in atom.update:
            read = [r_state[w] for w in term.read]
            results = z3_eval_itype(term.itype, read)
            for w, val in zip(term.write, results):
                r_state[w] = val
    return r_state[intf[0][1]][0]

# R(s): ranking of current state
R_s = eval_ranking([x_var, y_var, z_var])

# R(s'): ranking of next state
R_s_next = eval_ranking([x_next, y_var, z_var])

# Target predicate: x = y or x = z (current state)
at_target = z3.Or(x_var == y_var, x_var == z_var)

# Domain constraints: y=5, z=3, x in {0, 1, 2, 3, 4, 5}
domain = [y_var == 5, z_var == 3, z3.Or([x_var == i for i in range(6)])]

print("R(s) and R(s') built as Z3 expressions")

R(s) and R(s') built as Z3 expressions


### Verifying conditions 1 and 3

We negate each condition and check satisfiability. If Z3 returns `unsat`, the condition holds for all reachable states. If `sat`, the model is a concrete counterexample.

**Condition 1:** $R(s) \geq -1$ for all states (bounded from below)

In [9]:
# Condition 1: R(s) is bounded from below (R(s) >= -1)
solver = z3.Solver()
solver.add(*domain)
solver.add(R_s < -1)         # negate: R(s) < -1

result = solver.check()
if result == z3.unsat:
    print("VERIFIED: R(s) >= -1 for all reachable states")
else:
    m = solver.model()
    print(f"COUNTEREXAMPLE (condition 1): x={m[x_var]}, y={m[y_var]}, z={m[z_var]}")
    print(f"  R(s) = {float(m.eval(R_s, model_completion=True).as_fraction()):.6f} (should be >= -1)")

VERIFIED: R(s) >= -1 for all reachable states


**Condition 3:** $R(s') < R(s)$ when not at target (strict decrease)

In [10]:
# Condition 3: R(s') < R(s) when not at target
solver = z3.Solver()
solver.add(*domain)
solver.add(z3.Not(at_target))   # not at target
solver.add(R_s_next >= R_s)     # negate: R does not decrease

result = solver.check()
if result == z3.unsat:
    print("VERIFIED: R strictly decreases at all non-target states")
else:
    m = solver.model()
    print(f"COUNTEREXAMPLE (condition 3): x={m[x_var]}, y={m[y_var]}, z={m[z_var]}")
    r_val = float(m.eval(R_s, model_completion=True).as_fraction())
    r_next_val = float(m.eval(R_s_next, model_completion=True).as_fraction())
    print(f"  R(s) = {r_val:.6f}, R(s') = {r_next_val:.6f} (should be R(s') < R(s))")

VERIFIED: R strictly decreases at all non-target states


In [ ]:
from zrth.visual import show
stop = show(wrapped_counter)